In [1]:
# ================================================================
# CELL 1 — Imports & Configuration
# ================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler
import numpy as np
import matplotlib
matplotlib.use('Agg')   # non-interactive backend; change to 'TkAgg' if running interactively
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    confusion_matrix
)

# ── Hyperparameters ──────────────────────────────────────────────
OUTER_SPLITS = 5     # outer fold  → generates one unbiased test set each
INNER_SPLITS = 3     # inner fold  → model selection / threshold tuning
EPOCHS       = 5    # max epochs per inner fold
PATIENCE     = 5     # early-stopping patience (inner loop)
BATCH_SIZE   = 32
LR           = 5e-4
RANDOM_STATE = 45
PLOT_DIR     = "."   # where to save figures

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device          : {device}")
print(f"Outer folds     : {OUTER_SPLITS}")
print(f"Inner folds     : {INNER_SPLITS}")
print(f"Max epochs/fold : {EPOCHS}   Patience: {PATIENCE}")

Device          : cpu
Outer folds     : 5
Inner folds     : 3
Max epochs/fold : 5   Patience: 5


In [2]:
# ================================================================
# CELL 2 — Load Data
# ================================================================
X = np.load("X_features.npy")   # (N, 128, 94, 1)
y = np.load("y_labels.npy")     # (N,)

print(f"Dataset  →  X: {X.shape}  |  y: {y.shape}")
print(f"Classes  →  0 (real): {(y==0).sum()}  |  1 (fake): {(y==1).sum()}")

Dataset  →  X: (18105, 128, 94, 1)  |  y: (18105,)
Classes  →  0 (real): 7172  |  1 (fake): 10933


In [3]:
# ================================================================
# CELL 3 — Dataset & Model
# ================================================================

class AudioDataset(Dataset):
    """(N,H,W,C) numpy → (N,C,H,W) torch tensors."""
    def __init__(self, features, labels):
        self.X = torch.from_numpy(features).permute(0, 3, 1, 2).float()
        self.y = torch.from_numpy(labels).float().unsqueeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


class DeepfakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(64)
        self.pool  = nn.MaxPool2d(2, 2)
        self.ap    = nn.AdaptiveAvgPool2d((8, 8))
        self.fc    = nn.Sequential(
            nn.Linear(64*8*8, 128), nn.ReLU(), nn.Dropout(0.4), nn.Linear(128, 1)
        )
    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        return self.fc(torch.flatten(self.ap(x), 1))

print("AudioDataset and DeepfakeDetector defined.")

AudioDataset and DeepfakeDetector defined.


In [4]:
# ================================================================
# CELL 4 — Helper Functions
# ================================================================

def get_probs(model, loader):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for bx, by in loader:
            p = torch.sigmoid(model(bx.to(device))).cpu().numpy().ravel()
            probs.extend(p)
            labels.extend(by.numpy().ravel())
    return np.array(probs), np.array(labels)


def youden_threshold(probs, labels):
    """Threshold maximising Youden J = TPR - FPR."""
    fpr, tpr, thresholds = roc_curve(labels, probs)
    j = tpr - fpr
    idx = np.argmax(j)
    return float(thresholds[idx]), float(j[idx]), fpr, tpr


def evaluate(probs, labels, threshold):
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    return {
        "threshold"  : threshold,
        "accuracy"   : accuracy_score(labels, preds),
        "f1"         : f1_score(labels, preds, zero_division=0),
        "auc"        : roc_auc_score(labels, probs),
        "sensitivity": tp / (tp + fn + 1e-8),
        "specificity": tn / (tn + fp + 1e-8),
        "precision"  : tp / (tp + fp + 1e-8),
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
        "report"     : classification_report(labels, preds, digits=4),
    }


def train_one_fold(train_idx, val_idx, dataset, epochs, patience):
    """Train a fresh model; return best state dict and per-epoch history."""
    train_loader = DataLoader(dataset, batch_size=BATCH_SIZE,
                              sampler=SubsetRandomSampler(train_idx))
    val_loader   = DataLoader(dataset, batch_size=BATCH_SIZE,
                              sampler=SubsetRandomSampler(val_idx))

    model     = DeepfakeDetector().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.BCEWithLogitsLoss()

    best_auc, best_state = -1, None
    patience_ctr = 0
    history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_auc": []}

    for epoch in range(1, epochs + 1):
        # ── train ──
        model.train()
        t_loss = 0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward(); optimizer.step()
            t_loss += loss.item()
        history["train_loss"].append(t_loss / len(train_loader))

        # ── validate ──
        model.eval()
        v_loss = 0
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(device), by.to(device)
                v_loss += criterion(model(bx), by).item()
        history["val_loss"].append(v_loss / len(val_loader))

        vp, vl   = get_probs(model, val_loader)
        thresh, _, _, _ = youden_threshold(vp, vl)
        v_acc    = accuracy_score(vl, (vp >= thresh).astype(int))
        v_auc    = roc_auc_score(vl, vp)
        history["val_acc"].append(v_acc)
        history["val_auc"].append(v_auc)

        if v_auc > best_auc:
            best_auc   = v_auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                break

    return best_state, history, epoch   # epoch = actual epochs run


print("Helper functions defined.")

Helper functions defined.


In [5]:
# ================================================================
# CELL 5 — Nested K-Fold Training
#
# Structure:
#   Outer loop  (OUTER_SPLITS folds)  → held-out TEST set per fold
#   Inner loop  (INNER_SPLITS folds)  → trains model, tunes threshold
#                                        on TRAIN+VAL without touching TEST
# ================================================================

full_dataset = AudioDataset(X, y)

outer_skf = StratifiedKFold(n_splits=OUTER_SPLITS, shuffle=True, random_state=RANDOM_STATE)
inner_skf = StratifiedKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Storage for plotting and reporting
outer_results  = []          # final metrics per outer fold
roc_data       = []          # (fpr, tpr, auc, youden_point) per outer fold
pr_data        = []          # (recall, precision, ap) per outer fold
all_histories  = []          # loss/acc curves: list of lists (inner folds)
all_test_probs = []
all_test_labels= []
agg_cm         = np.zeros((2, 2), dtype=int)

best_global_auc   = -1
best_model_state  = None
best_outer_fold   = None
best_youden_thresh = 0.5

print("=" * 70)
print(f"  NESTED K-FOLD  |  Outer: {OUTER_SPLITS}  |  Inner: {INNER_SPLITS}")
print(f"  Total inner training runs: {OUTER_SPLITS * INNER_SPLITS}")
print("=" * 70)

for outer_idx, (trainval_idx, test_idx) in enumerate(
        outer_skf.split(X, y), start=1):

    print(f"\n{'━'*70}")
    print(f"  OUTER FOLD {outer_idx}/{OUTER_SPLITS}  "
          f"  trainval={len(trainval_idx)}  test={len(test_idx)}")
    print(f"{'━'*70}")

    X_trainval = X[trainval_idx]
    y_trainval = y[trainval_idx]

    # ── Inner loop: find best model + Youden threshold ────────────
    inner_best_auc    = -1
    inner_best_state  = None
    inner_best_thresh = 0.5
    outer_fold_histories = []

    for inner_idx, (tr_idx_rel, val_idx_rel) in enumerate(
            inner_skf.split(X_trainval, y_trainval), start=1):

        # Indices relative to full dataset
        tr_idx  = trainval_idx[tr_idx_rel]
        val_idx = trainval_idx[val_idx_rel]

        print(f"  ├─ Inner fold {inner_idx}/{INNER_SPLITS}  "
              f"train={len(tr_idx)}  val={len(val_idx)}")

        state, history, epochs_ran = train_one_fold(
            tr_idx, val_idx, full_dataset, EPOCHS, PATIENCE)
        outer_fold_histories.append(history)

        # Evaluate inner val with Youden threshold
        model = DeepfakeDetector().to(device)
        model.load_state_dict(state)
        val_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE,
                                sampler=SubsetRandomSampler(val_idx))
        vp, vl = get_probs(model, val_loader)
        thresh, j_stat, _, _ = youden_threshold(vp, vl)
        val_auc = roc_auc_score(vl, vp)

        print(f"  │    Epochs: {epochs_ran:2d}  "
              f"Val AUC: {val_auc:.4f}  "
              f"Youden thresh: {thresh:.4f}  "
              f"J: {j_stat:.4f}")

        if val_auc > inner_best_auc:
            inner_best_auc    = val_auc
            inner_best_state  = state
            inner_best_thresh = thresh

    all_histories.append(outer_fold_histories)

    # ── Evaluate on held-out TEST set ─────────────────────────────
    model = DeepfakeDetector().to(device)
    model.load_state_dict(inner_best_state)
    test_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE,
                             sampler=SubsetRandomSampler(test_idx))
    test_probs, test_labels = get_probs(model, test_loader)

    metrics = evaluate(test_probs, test_labels, inner_best_thresh)
    metrics["outer_fold"]   = outer_idx
    metrics["inner_thresh"] = inner_best_thresh
    outer_results.append(metrics)

    # Accumulate for aggregate plots
    all_test_probs.extend(test_probs)
    all_test_labels.extend(test_labels)
    agg_cm += confusion_matrix(test_labels,
                               (test_probs >= inner_best_thresh).astype(int))

    # ROC data
    fpr_r, tpr_r, _ = roc_curve(test_labels, test_probs)
    fold_auc = auc(fpr_r, tpr_r)
    # Youden point on test ROC
    j_vals = tpr_r - fpr_r
    yp_idx = np.argmax(j_vals)
    roc_data.append((fpr_r, tpr_r, fold_auc,
                     fpr_r[yp_idx], tpr_r[yp_idx]))

    # PR data
    prec_r, rec_r, _ = precision_recall_curve(test_labels, test_probs)
    ap = average_precision_score(test_labels, test_probs)
    pr_data.append((rec_r, prec_r, ap))

    print(f"\n  ✦ Outer {outer_idx} TEST  thresh={inner_best_thresh:.4f}  "
          f"Acc={metrics['accuracy']:.4f}  F1={metrics['f1']:.4f}  "
          f"AUC={metrics['auc']:.4f}")

    if metrics["auc"] > best_global_auc:
        best_global_auc    = metrics["auc"]
        best_model_state   = inner_best_state
        best_outer_fold    = outer_idx
        best_youden_thresh = inner_best_thresh

print("\n" + "=" * 70)
print(f"  All {OUTER_SPLITS} outer folds complete.")
print("=" * 70)

  NESTED K-FOLD  |  Outer: 5  |  Inner: 3
  Total inner training runs: 15

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  OUTER FOLD 1/5    trainval=14484  test=3621
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ├─ Inner fold 1/3  train=9656  val=4828
  │    Epochs:  5  Val AUC: 0.9613  Youden thresh: 0.5564  J: 0.7793
  ├─ Inner fold 2/3  train=9656  val=4828
  │    Epochs:  5  Val AUC: 0.9642  Youden thresh: 0.6334  J: 0.7983
  ├─ Inner fold 3/3  train=9656  val=4828
  │    Epochs:  5  Val AUC: 0.9865  Youden thresh: 0.6640  J: 0.8884

  ✦ Outer 1 TEST  thresh=0.6640  Acc=0.9409  F1=0.9512  AUC=0.9862

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  OUTER FOLD 2/5    trainval=14484  test=3621
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ├─ Inner fold 1/3  train=9656  val=4828
  │    Epochs:  5  Val AUC: 0.9752  Youden thresh: 0.4235  J: 0.8396
  ├─ Inner fold 2/3  train=9656  val

In [6]:
# ================================================================
# CELL 6 — Summary Metrics Table
# ================================================================

keys = ["accuracy", "f1", "auc", "sensitivity", "specificity", "threshold"]

print("\n" + "=" * 70)
print("  NESTED K-FOLD  —  TEST SET RESULTS")
print("=" * 70)
print(f"  Outer folds : {OUTER_SPLITS}  |  Inner folds : {INNER_SPLITS}")
print(f"  Total inner runs : {OUTER_SPLITS * INNER_SPLITS}")
print(f"  Threshold method : Youden's J  (tuned on inner val, applied to outer test)")
print()

hdr = (f"{'Fold':>5} {'Thresh':>8} {'Acc':>8} {'F1':>8} "
       f"{'AUC':>8} {'Sens':>8} {'Spec':>8}")
print(hdr)
print("-" * len(hdr))
for r in outer_results:
    print(f"{r['outer_fold']:>5} {r['threshold']:>8.4f} {r['accuracy']:>8.4f} "
          f"{r['f1']:>8.4f} {r['auc']:>8.4f} "
          f"{r['sensitivity']:>8.4f} {r['specificity']:>8.4f}")
print("-" * len(hdr))

for k in keys:
    vals = [r[k] for r in outer_results]
    print(f"  {k:<14} mean={np.mean(vals):.4f}  std={np.std(vals):.4f}  "
          f"min={np.min(vals):.4f}  max={np.max(vals):.4f}")

print()
print(f"  Best outer fold (AUC) : Fold {best_outer_fold}  (AUC={best_global_auc:.4f})")
print(f"  Best Youden threshold : {best_youden_thresh:.4f}")

print("\nAggregated Classification Report (all outer test sets):")
all_p = np.array(all_test_probs)
all_l = np.array(all_test_labels)
mean_thresh = np.mean([r["threshold"] for r in outer_results])
print(classification_report(all_l, (all_p >= mean_thresh).astype(int), digits=4))


  NESTED K-FOLD  —  TEST SET RESULTS
  Outer folds : 5  |  Inner folds : 3
  Total inner runs : 15
  Threshold method : Youden's J  (tuned on inner val, applied to outer test)

 Fold   Thresh      Acc       F1      AUC     Sens     Spec
-----------------------------------------------------------
    1   0.6640   0.9409   0.9512   0.9862   0.9543   0.9205
    2   0.5401   0.9224   0.9348   0.9776   0.9214   0.9240
    3   0.4598   0.9265   0.9378   0.9821   0.9172   0.9407
    4   0.4667   0.9050   0.9217   0.9683   0.9263   0.8725
    5   0.8108   0.8964   0.9119   0.9698   0.8879   0.9094
-----------------------------------------------------------
  accuracy       mean=0.9183  std=0.0158  min=0.8964  max=0.9409
  f1             mean=0.9315  std=0.0136  min=0.9119  max=0.9512
  auc            mean=0.9768  std=0.0069  min=0.9683  max=0.9862
  sensitivity    mean=0.9214  std=0.0212  min=0.8879  max=0.9543
  specificity    mean=0.9134  std=0.0228  min=0.8725  max=0.9407
  threshold      

In [7]:
# ================================================================
# CELL 7 — Plot 1: ROC Curves (per fold + mean)
# ================================================================
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import interp1d

fig, ax = plt.subplots(figsize=(7, 6))
mean_fpr = np.linspace(0, 1, 300)
tprs_interp = []

palette = plt.cm.tab10.colors

for i, (fpr, tpr, fold_auc, yp_fpr, yp_tpr) in enumerate(roc_data):
    ax.plot(fpr, tpr, lw=1.2, alpha=0.45, color=palette[i],
            label=f"Fold {i+1}  AUC={fold_auc:.3f}")
    ax.scatter(yp_fpr, yp_tpr, marker='x', s=60, color=palette[i],
               zorder=5, linewidths=1.8)
    f_interp = interp1d(fpr, tpr, kind='linear',
                        bounds_error=False, fill_value=(0, 1))
    tprs_interp.append(f_interp(mean_fpr))

mean_tpr  = np.mean(tprs_interp, axis=0)
std_tpr   = np.std(tprs_interp, axis=0)
mean_auc  = np.mean([d[2] for d in roc_data])
std_auc   = np.std([d[2] for d in roc_data])

ax.plot(mean_fpr, mean_tpr, color='black', lw=2.5,
        label=f"Mean  AUC={mean_auc:.3f} ± {std_auc:.3f}")
ax.fill_between(mean_fpr, mean_tpr - std_tpr, mean_tpr + std_tpr,
                color='grey', alpha=0.18, label="± 1 std")
ax.plot([0, 1], [0, 1], 'k--', lw=0.8, alpha=0.5, label="Random")

# Youden marker legend note
ax.scatter([], [], marker='x', s=60, color='grey', linewidths=1.8,
           label="Youden point (per fold)")

ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves — Nested K-Fold (Outer Test Sets)", fontsize=13, fontweight='bold')
ax.legend(loc="lower right", fontsize=8.5, framealpha=0.9)
ax.set_xlim(-0.01, 1.01); ax.set_ylim(-0.01, 1.01)
ax.grid(True, linestyle='--', alpha=0.35)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/plot1_roc_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → plot1_roc_curves.png")

Saved → plot1_roc_curves.png


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_19972\3848052740.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# ================================================================
# CELL 8 — Plot 2: Precision-Recall Curves
# ================================================================
from sklearn.metrics import auc as sk_auc

fig, ax = plt.subplots(figsize=(7, 6))
baseline = (all_l == 1).mean()   # positive class prevalence

mean_rec  = np.linspace(0, 1, 300)
precs_interp = []

for i, (rec, prec, ap) in enumerate(pr_data):
    ax.plot(rec, prec, lw=1.2, alpha=0.45, color=palette[i],
            label=f"Fold {i+1}  AP={ap:.3f}")
    f_interp = interp1d(rec[::-1], prec[::-1], kind='linear',
                        bounds_error=False,
                        fill_value=(prec[np.argmin(rec)], prec[0]))
    precs_interp.append(f_interp(mean_rec))

mean_prec = np.mean(precs_interp, axis=0)
std_prec  = np.std(precs_interp, axis=0)
mean_ap   = np.mean([d[2] for d in pr_data])
std_ap    = np.std([d[2] for d in pr_data])

ax.plot(mean_rec, mean_prec, color='black', lw=2.5,
        label=f"Mean  AP={mean_ap:.3f} ± {std_ap:.3f}")
ax.fill_between(mean_rec, mean_prec - std_prec, mean_prec + std_prec,
                color='grey', alpha=0.18, label="± 1 std")
ax.axhline(baseline, color='red', lw=1, linestyle='--',
           label=f"Baseline (prevalence={baseline:.2f})")

ax.set_xlabel("Recall", fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.set_title("Precision-Recall Curves — Nested K-Fold", fontsize=13, fontweight='bold')
ax.legend(loc="upper right", fontsize=8.5, framealpha=0.9)
ax.set_xlim(-0.01, 1.01); ax.set_ylim(-0.01, 1.05)
ax.grid(True, linestyle='--', alpha=0.35)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/plot2_pr_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → plot2_pr_curves.png")

In [8]:
# ================================================================
# CELL 9 — Plot 3: Aggregated Confusion Matrix Heatmap
# ================================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# ── Left: raw counts ──────────────────────────────────────────────
cmap_counts = LinearSegmentedColormap.from_list(
    "wbl", ["#ffffff", "#4a90d9", "#1a3a5c"])
sns.heatmap(agg_cm, annot=True, fmt='d', cmap=cmap_counts,
            xticklabels=['Pred Real', 'Pred Fake'],
            yticklabels=['True Real', 'True Fake'],
            linewidths=0.5, linecolor='#cccccc',
            annot_kws={'size': 16, 'weight': 'bold'},
            ax=axes[0])
axes[0].set_title("Aggregated CM — Counts", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Predicted", fontsize=11)
axes[0].set_ylabel("Actual", fontsize=11)

# ── Right: row-normalised (rates) ─────────────────────────────────
cm_norm = agg_cm.astype(float) / agg_cm.sum(axis=1, keepdims=True)
cmap_norm = LinearSegmentedColormap.from_list(
    "wgr", ["#ffffff", "#66c266", "#145214"])
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap=cmap_norm,
            xticklabels=['Pred Real', 'Pred Fake'],
            yticklabels=['True Real', 'True Fake'],
            vmin=0, vmax=1,
            linewidths=0.5, linecolor='#cccccc',
            annot_kws={'size': 14},
            ax=axes[1])
axes[1].set_title("Aggregated CM — Row-Normalised Rates", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Predicted", fontsize=11)
axes[1].set_ylabel("Actual", fontsize=11)

# Label corner cells
for ax, fmt in zip(axes, ['d', '.3f']):
    ax.texts[0].set_text(f"TN\n{ax.texts[0].get_text()}")
    ax.texts[3].set_text(f"TP\n{ax.texts[3].get_text()}")

plt.suptitle(f"Confusion Matrix — Nested {OUTER_SPLITS}-Fold  "
             f"(Youden threshold per fold)",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/plot3_confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → plot3_confusion_matrix.png")
print(f"  Aggregated CM:\n{agg_cm}")

Saved → plot3_confusion_matrix.png
  Aggregated CM:
[[ 6551   621]
 [  859 10074]]


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_19972\1368356886.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# ================================================================
# CELL 10 — Plot 4: Loss & Accuracy Curves (per inner fold)
#
# Shows train_loss, val_loss, val_acc, val_auc for each inner fold
# within each outer fold.  Layout: OUTER_SPLITS rows × INNER_SPLITS cols.
# ================================================================

n_rows = OUTER_SPLITS
n_cols = INNER_SPLITS
fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(n_cols * 4.5, n_rows * 3.2),
                         squeeze=False)

for o_idx, outer_fold_histories in enumerate(all_histories):
    for i_idx, hist in enumerate(outer_fold_histories):
        ax = axes[o_idx][i_idx]
        ep = range(1, len(hist["train_loss"]) + 1)

        color_tl = '#e05c5c'   # train loss
        color_vl = '#5c8fe0'   # val loss
        color_va = '#2ca02c'   # val acc
        color_au = '#9467bd'   # val auc

        ax2 = ax.twinx()       # second y-axis for acc / auc

        ax.plot(ep, hist["train_loss"], color=color_tl, lw=1.5,
                label="Train Loss", linestyle='-')
        ax.plot(ep, hist["val_loss"],   color=color_vl, lw=1.5,
                label="Val Loss",   linestyle='--')
        ax2.plot(ep, hist["val_acc"],   color=color_va, lw=1.5,
                 label="Val Acc",   linestyle='-.')
        ax2.plot(ep, hist["val_auc"],   color=color_au, lw=1.5,
                 label="Val AUC",   linestyle=':')

        ax.set_title(f"Outer {o_idx+1} — Inner {i_idx+1}",
                     fontsize=9, fontweight='bold')
        ax.set_xlabel("Epoch", fontsize=8)
        ax.set_ylabel("Loss", fontsize=8, color='grey')
        ax2.set_ylabel("Acc / AUC", fontsize=8, color='grey')
        ax2.set_ylim(0, 1.05)
        ax.tick_params(labelsize=7)
        ax2.tick_params(labelsize=7)

        # Unified legend
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2,
                  fontsize=6.5, loc='upper right', framealpha=0.7)
        ax.grid(True, linestyle='--', alpha=0.3)

plt.suptitle(
    f"Training Curves — All Inner Folds  "
    f"(Outer {OUTER_SPLITS} × Inner {INNER_SPLITS} = "
    f"{OUTER_SPLITS*INNER_SPLITS} runs)",
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/plot4_training_curves.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved → plot4_training_curves.png")

Saved → plot4_training_curves.png


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_19972\1240216985.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# ================================================================
# CELL 11 — Save Best Model
# ================================================================

SAVE_PATH = "deepfake_logmel_nested_kfold.pth"

torch.save(
    {
        "model_state_dict" : best_model_state,
        "best_outer_fold"  : best_outer_fold,
        "best_auc"         : best_global_auc,
        "youden_threshold" : best_youden_thresh,
        "outer_splits"     : OUTER_SPLITS,
        "inner_splits"     : INNER_SPLITS,
        "outer_results"    : [
            {k: v for k, v in r.items() if k != "report"}
            for r in outer_results
        ],
    },
    SAVE_PATH,
)

print(f"✅ Model saved → {SAVE_PATH}")
print(f"   Best outer fold  : {best_outer_fold}")
print(f"   Best AUC (test)  : {best_global_auc:.4f}")
print(f"   Youden threshold : {best_youden_thresh:.4f}")
print()
print("To reload:")
print("  ckpt = torch.load('deepfake_logmel_nested_kfold.pth')")
print("  model.load_state_dict(ckpt['model_state_dict'])")
print("  threshold = ckpt['youden_threshold']")

✅ Model saved → deepfake_logmel_nested_kfold.pth
   Best outer fold  : 1
   Best AUC (test)  : 0.9862
   Youden threshold : 0.6640

To reload:
  ckpt = torch.load('deepfake_logmel_nested_kfold.pth')
  model.load_state_dict(ckpt['model_state_dict'])
  threshold = ckpt['youden_threshold']
